# EDA - Forecasting Financial Inclusion in Ethiopia

In [21]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

import sys
import os
sys.path.append(os.path.abspath('../'))

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

In [5]:
df_data = pd.read_excel('../data/raw/ethiopia_fi_unified_data.xlsx', sheet_name='ethiopia_fi_unified_data')
df_data.head(3)
df_data.shape

(43, 34)

In [6]:
df_data.head(3)

,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,...,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Baseline year,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,...,NaN,NaN,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN


In [4]:
df_data.describe()

,value_numeric,observation_date,period_start,period_end,region,related_indicator,relationship_type,impact_direction,impact_magnitude,impact_estimate,lag_months,evidence_basis,collected_by,notes
count,3.300000e+01,43,10,10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,43,0.0
mean,9.437258e+10,2024-05-09 02:13:57.209302272,2024-02-28 04:48:00,2025-05-08 19:12:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-20 00:00:00,NaN
min,1.080000e+00,2014-12-31 00:00:00,2021-09-01 00:00:00,2024-07-07 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-20 00:00:00,NaN
25%,2.400000e+01,2023-07-16 00:00:00,2024-07-08 00:00:00,2025-07-01 18:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-20 00:00:00,NaN
50%,6.140000e+01,2024-12-31 00:00:00,2024-07-08 00:00:00,2025-07-07 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-20 00:00:00,NaN
75%,1.500000e+07,2025-07-07 00:00:00,2024-07-08 00:00:00,2025-07-07 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-20 00:00:00,NaN
max,2.380000e+12,2030-12-31 00:00:00,2024-10-15 00:00:00,2025-07-07 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-01-20 00:00:00,NaN
std,4.231061e+11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 34 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   record_id            43 non-null     object        
 1   record_type          43 non-null     object        
 2   category             10 non-null     object        
 3   pillar               33 non-null     object        
 4   indicator            43 non-null     object        
 5   indicator_code       43 non-null     object        
 6   indicator_direction  33 non-null     object        
 7   value_numeric        33 non-null     float64       
 8   value_text           10 non-null     object        
 9   value_type           43 non-null     object        
 10  unit                 33 non-null     object        
 11  observation_date     43 non-null     datetime64[ns]
 12  period_start         10 non-null     datetime64[ns]
 13  period_end           10 non-null     

In [8]:
df_data.dtypes

record_id                      object
record_type                    object
category                       object
pillar                         object
indicator                      object
indicator_code                 object
indicator_direction            object
value_numeric                 float64
value_text                     object
value_type                     object
unit                           object
observation_date       datetime64[ns]
period_start           datetime64[ns]
period_end             datetime64[ns]
fiscal_year                    object
gender                         object
location                       object
region                        float64
source_name                    object
source_type                    object
source_url                     object
confidence                     object
related_indicator             float64
relationship_type             float64
impact_direction              float64
impact_magnitude              float64
impact_estim

In [10]:
print(f"Data Observation date Timeframe: {df_data['observation_date'].min()} to {df_data['observation_date'].max()}")

Data Observation date Timeframe: 2014-12-31 00:00:00 to 2030-12-31 00:00:00


## DATA EXPLORATION

In [12]:
print("\n[1] Record Counts by key groupings:")
for col in ['record_type', 'pillar', 'source_type', 'confidence']:
    if col in df_data.columns and df_data[col].notna().any():
        print(f"\n--- Count by {col} ---")
        print(df_data[col].value_counts())


[1] Record Counts by key groupings:

--- Count by record_type ---
record_type
observation    30
event          10
target          3
Name: count, dtype: int64

--- Count by pillar ---
pillar
ACCESS           16
USAGE            11
GENDER            5
AFFORDABILITY     1
Name: count, dtype: int64

--- Count by source_type ---
source_type
operator      15
survey        10
regulator      7
research       4
policy         3
calculated     2
news           2
Name: count, dtype: int64

--- Count by confidence ---
confidence
high      40
medium     3
Name: count, dtype: int64


In [13]:
print("\n[2] Temporal Range of Observations:")
if 'observation_date' in df_data.columns:
    obs_dates = pd.to_datetime(df_data[df_data['record_type'] == 'observation']['observation_date'], errors='coerce')
    print(f"Earliest Observation: {obs_dates.min()}")
    print(f"Latest Observation:   {obs_dates.max()}")


[2] Temporal Range of Observations:
Earliest Observation: 2014-12-31 00:00:00
Latest Observation:   2025-12-31 00:00:00


In [14]:
print("\n[3] Unique Indicators and Coverage:")
if 'indicator_code' in df_data.columns:
    print(df_data.groupby('indicator_code').size().reset_index(name='record_count'))


[3] Unique Indicators and Coverage:
        indicator_code  record_count
0           ACC_4G_COV             2
1            ACC_FAYDA             4
2       ACC_MM_ACCOUNT             2
3       ACC_MOBILE_PEN             1
4        ACC_OWNERSHIP             7
5      AFF_DATA_INCOME             1
6        EVT_CROSSOVER             1
7         EVT_ETHIOPAY             1
8            EVT_FAYDA             1
9        EVT_FX_REFORM             1
10           EVT_MPESA             1
11   EVT_MPESA_INTEROP             1
12           EVT_NFIS2             1
13       EVT_SAFARICOM             1
14    EVT_SAFCOM_PRICE             1
15        EVT_TELEBIRR             1
16         GEN_GAP_ACC             2
17      GEN_GAP_MOBILE             1
18        GEN_MM_SHARE             2
19     USG_ACTIVE_RATE             1
20       USG_ATM_COUNT             1
21       USG_ATM_VALUE             1
22       USG_CROSSOVER             1
23    USG_MPESA_ACTIVE             1
24     USG_MPESA_USERS             1
2

In [15]:
print("\n[4] Cataloged Events:")
events = df_data[df_data['record_type'] == 'event']
if not events.empty:
    print(events[['record_id', 'category', 'notes']])
else:
    print("No events found in baseline dataset.")


[4] Cataloged Events:
   record_id        category  notes
33  EVT_0001  product_launch    NaN
34  EVT_0002    market_entry    NaN
35  EVT_0003  product_launch    NaN
36  EVT_0004  infrastructure    NaN
37  EVT_0005          policy    NaN
38  EVT_0006       milestone    NaN
39  EVT_0007     partnership    NaN
40  EVT_0008  infrastructure    NaN
41  EVT_0009          policy    NaN
42  EVT_0010         pricing    NaN


In [17]:
df_impact = pd.read_excel('../data/raw/ethiopia_fi_unified_data.xlsx', sheet_name='Impact_sheet')

In [ ]:
df_impact.head(3)

In [18]:
print("\n[5] Reviewing Existing Impact Links:")
print(f"Total existing impact links tracked: {len(df_impact)}")


[5] Reviewing Existing Impact Links:
Total existing impact links tracked: 14


## ENRICH THE DATASET

In [19]:
def get_next_id(df, prefix="REC_"):
    if df.empty or 'record_id' not in df.columns or df['record_id'].isna().all():
        return f"{prefix}0006"
    existing_ids = df['record_id'].dropna().str.extract(r'(\d+)').astype(int)
    if existing_ids.empty:
        return f"{prefix}0006"
    return f"{prefix}{str(existing_ids.max().values[0] + 1).zfill(4)}"

new_records = []

In [22]:
new_records.append({
    "record_id": get_next_id(df_data),
    "record_type": "observation",
    "pillar": "ACCESS",
    "indicator": "Mobile Money Accounts",
    "indicator_code": "MOBILE_MONEY_ACC",
    "value_numeric": 43000000,
    "observation_date": "2024-02-01",
    "source_name": "Ethio Telecom Performance Report",
    "source_url": "https://www.ethiotelecom.et",
    "confidence": "high",
    "collected_by": "Data_Enrichment_Script",
    "collection_date": datetime.today().strftime('%Y-%m-%d'),
    "original_text": "Telebirr subscribers reached 43 million by early 2024.",
    "notes": "Critical infrastructure baseline tracking the exponential growth of mobile banking alternatives over traditional brick-and-mortar access."
})

In [23]:
# --- Addition 2: New Event (National Bank of Ethiopia Directive on Digital Financial Services) ---
rec_id_event = f"REC_{str(int(get_next_id(df_data).split('_')[1])+1).zfill(4)}"
new_records.append({
    "record_id": rec_id_event,
    "record_type": "event",
    "category": "policy",
    "pillar": np.nan, # explicitly left blank per schema guidelines
    "observation_date": "2020-04-01",
    "source_name": "National Bank of Ethiopia Directives",
    "source_url": "https://nbe.gov.et",
    "confidence": "high",
    "collected_by": "Data_Enrichment_Script",
    "collection_date": datetime.today().strftime('%Y-%m-%d'),
    "original_text": "NBE issued Directive No. ONPSD/01/2020 allowing non-banking institutions to offer mobile money services.",
    "notes": "Regulatory milestone enabling Safaricom (M-Pesa) and Ethio Telecom (Telebirr) entry into the financial ecosystem."
})

In [24]:
# Append new data rows to master data sheet
df_new_rows = pd.DataFrame(new_records)
df_data_updated = pd.concat([df_data, df_new_rows], ignore_index=True)

In [25]:
# --- Addition 3: New Impact Link mapping back to the Policy Event ---
new_impacts = []
new_impacts.append({
    "parent_id": rec_id_event,
    "pillar": "USAGE",
    "related_indicator": "MOBILE_MONEY_ACC",
    "impact_direction": "positive",
    "impact_magnitude": "high",
    "lag_months": 12,
    "evidence_basis": "Empirical historical trajectory of analog mobile money policy shifts across East Africa (e.g., Kenya M-Pesa case study)."
})

In [26]:
df_impact_updated = pd.concat([df_impact, pd.DataFrame(new_impacts)], ignore_index=True)

In [27]:
# ==========================================
# 4. SAVE EXCEL UPDATES & DATA LOG
# ==========================================
# Export to Excel
with pd.ExcelWriter('../data/processed/ethiopia_fi_unified_data_updated.xlsx', engine='openpyxl') as writer:
    df_data_updated.to_excel(writer, sheet_name='data', index=False)
    df_impact_updated.to_excel(writer, sheet_name='impact_links', index=False)

print("\n✓ Exported updated dataset to 'ethiopia_fi_unified_data_updated.xlsx'.")


✓ Exported updated dataset to 'ethiopia_fi_unified_data_updated.xlsx'.


In [28]:
# Generate data_enrichment_log.md
log_content = f"""# Data Enrichment Log

**Author / Collected By:** Data_Enrichment_Script  
**Date:** {datetime.today().strftime('%Y-%m-%d')}  

## Overview of Changes
This log details structural additions made to track financial inclusion variables targeting forecasting models for Access and Usage in Ethiopia.

---

## 1. New Observations Added
* **Indicator Code:** `MOBILE_MONEY_ACC`
* **Pillar:** ACCESS
* **Value:** 43,000,000
* **Date:** 2024-02-01
* **Source:** Ethio Telecom Performance Report
* **Source URL:** https://www.ethiotelecom.et
* **Original Quote:** "Telebirr subscribers reached 43 million by early 2024."
* **Confidence Level:** High
* **Strategic Value:** Provides modern high-velocity infrastructure data points necessary for non-linear forecasting models.

---

## 2. New Events Tracked
* **Category:** Policy
* **Event Detail:** National Bank of Ethiopia Directive No. ONPSD/01/2020
* **Date:** 2020-04-01
* **Source:** National Bank of Ethiopia Directives
* **Source URL:** https://nbe.gov.et
* **Original Quote:** "NBE issued Directive No. ONPSD/01/2020 allowing non-banking institutions to offer mobile money services."
* **Confidence Level:** High
* **Strategic Value:** Flags the historical starting point of structural regulatory changes causing a break from linear baseline growth projections.

---

## 3. New Impact Links Modeled
* **Parent ID Reference:** {rec_id_event} (NBE Directive Event)
* **Target Indicator Linkage:** `MOBILE_MONEY_ACC`
* **Pillar Alignment:** USAGE
* **Direction / Magnitude:** Positive / High
* **Modeled Lag Horizon:** 12 Months
* **Evidence Justification:** Directly correlates systemic expansion to structural regulatory policy relaxation based on broader regional deployment precedents.
"""

with open("data_enrichment_log.md", "w", encoding="utf-8") as f:
    f.write(log_content)

print("✓ Generated 'data_enrichment_log.md'.")

# --- Git workflow instructions for terminal use ---
print("\n=== NEXT STEPS FOR PR COMPLETION ===")
print("Run the following commands in your local bash shell environment:")
print("git add financial_inclusion_data_updated.xlsx data_enrichment_log.md")
print("git commit -m 'feat: complete data exploration and schema-compliant enrichment for task-1'")
print("git push origin task-1")
print("Then log into your repository hosting service to open a Pull Request (PR) merging 'task-1' into 'main'.")

✓ Generated 'data_enrichment_log.md'.

=== NEXT STEPS FOR PR COMPLETION ===
Run the following commands in your local bash shell environment:
git add financial_inclusion_data_updated.xlsx data_enrichment_log.md
git commit -m 'feat: complete data exploration and schema-compliant enrichment for task-1'
git push origin task-1
Then log into your repository hosting service to open a Pull Request (PR) merging 'task-1' into 'main'.
